# ModelShield — Phase 1: An AI Firewall for Adversarial Attack Detection & Defense
**Repository:** [https://github.com/Adi-015/ModelShield.git](https://github.com/Adi-015/ModelShield.git)  
**Author:** Aditya Sharma  
**Task:** Robust Deep Learning Security Pipeline on CIFAR-10

---

### Project Architecture & Objectives
1. **Module 1 (Core Classifier):** Custom Convolutional Neural Network trained on CIFAR-10.
2. **Module 2 (Attack Engine):** Fast Gradient Sign Method (FGSM) adversarial attack generation across perturbation magnitudes ($\\epsilon \\in [0.00, 0.01, 0.03, 0.05, 0.10]$).
3. **Module 3 (AI Firewall Detector):** Binary CNN detector that screens incoming inputs for adversarial artifacts before classifier inference.
4. **Module 4 (Integrated Security Pipeline):** Pre-inference gateway that routes verified clean inputs to the classifier while dropping/quarantining malicious adversarial inputs.

---

### Historical Baseline Reference (Previously Recorded Run)
*Note: The results below were recorded from a previous Google Colab session where weights were stored in ephemeral runtime storage. Checkpoints must be reproduced to regenerate actual model weights.*

| Component | Metric | Historical Recorded Result |
| :--- | :--- | :---: |
| **Classifier (15 Epochs)** | Final Train Accuracy | 70.95% |
| | Final Val Accuracy | 78.38% |
| | Final Train Loss | 0.8344 |
| | Final Val Loss | 0.6079 |
| | Best Val Checkpoint | Epoch 15 |
| **Detector (10 Epochs)** | Final Train Accuracy | 99.52% |
| | Best Val Accuracy | 99.92% |
| | Final Val Accuracy | 99.46% |
| | Final Train Loss | 0.0158 |
| | Final Val Loss | 0.0173 |
| | Best Val Checkpoint | Epoch 5 |
| **FGSM Attack ($\\epsilon=0.00$)** | Adversarial Accuracy / ASR | 79.12% / 0.00% |
| **FGSM Attack ($\\epsilon=0.01$)** | Adversarial Accuracy / ASR | 22.26% / 71.87% |
| **FGSM Attack ($\\epsilon=0.03$)** | Adversarial Accuracy / ASR | 6.36% / 92.01% |


## 1. Permanent Google Drive Checkpoint & Storage Setup
This notebook integrates Google Drive persistence under `/content/drive/MyDrive/ModelShield/`.
- Checkpoints are saved both as `latest` (after every epoch for resumption) and `best` (when validation accuracy improves).
- If running in Google Colab, Drive is mounted to guarantee that trained weights and experiment logs survive session disconnections.
- If running locally, storage defaults cleanly to `./checkpoints`, `./outputs`, and `./logs`.


In [ ]:
import os
import sys
from pathlib import Path

# 1. Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("[✓] Google Drive mounted successfully at /content/drive")
except ImportError:
    IN_COLAB = False
    print("[i] Not running in Google Colab. Using local repository storage.")

# 2. Configure permanent storage paths
if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive/ModelShield')
    if not Path('/content/drive/MyDrive').exists():
        raise RuntimeError(
            "Google Drive was requested, but /content/drive/MyDrive is inaccessible. "
            "Please grant Drive permissions when prompted."
        )
    STORAGE_ROOT = DRIVE_ROOT
else:
    STORAGE_ROOT = Path('.').resolve()

# Subdirectories
CHECKPOINT_DIR = STORAGE_ROOT / "checkpoints"
OUTPUTS_DIR = STORAGE_ROOT / "outputs"
LOGS_DIR = STORAGE_ROOT / "logs"

for directory in (CHECKPOINT_DIR, OUTPUTS_DIR, LOGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Active Storage Root: {STORAGE_ROOT}")
print(f"  ├── Checkpoints: {CHECKPOINT_DIR}")
print(f"  ├── Outputs:     {OUTPUTS_DIR}")
print(f"  └── Logs:        {LOGS_DIR}")


## 2. Hardware Acceleration & CUDA Diagnostics
Detect GPU hardware (NVIDIA T4 / V100 / A100) or fall back gracefully to CPU.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device Selection] Using compute device: {device}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Model:       {props.name}")
    print(f"  Compute Cap:     {props.major}.{props.minor}")
    print(f"  Total VRAM:      {props.total_memory / (1024**3):.2f} GB")
    print(f"  PyTorch CUDA:    {torch.version.cuda}")
else:
    print("  Notice: Running on CPU. Training will be functional but slower than GPU acceleration.")


## 3. Clone Repository & Setup Modular Environment
Ensures the modular source code from [https://github.com/Adi-015/ModelShield.git](https://github.com/Adi-015/ModelShield.git) is loaded onto `sys.path`.


In [ ]:
# If running directly inside a fresh Colab instance, clone the official repository:
if IN_COLAB and not Path('src').exists():
    print("Cloning ModelShield repository from GitHub...")
    !git clone https://github.com/Adi-015/ModelShield.git
    %cd ModelShield

# Add repository root to Python module search path
REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Sync config paths with permanent storage
import config
config.setup_storage(custom_root=STORAGE_ROOT)
print("[✓] Modular codebase loaded. config.STORAGE_ROOT aligned with:", config.STORAGE_ROOT)


## 4. Deterministic Seed Setting & Framework Alignment
Set random seeds across Python, NumPy, and PyTorch (seed = 42) for exact reproducibility.


In [ ]:
config.set_seed(config.RANDOM_SEED)
print(f"[✓] Deterministic seeds configured (RANDOM_SEED = {config.RANDOM_SEED}). cuDNN deterministic flags engaged.")


## 5. CIFAR-10 Ingestion, Preprocessing & Non-Leaking Splits
- Training Set: 40,000 images
- Validation Set: 10,000 images (used for model tuning and selection)
- Test Set: 10,000 images (untouched, reserved for final evaluation)
- Normalization: Mean (0.4914, 0.4822, 0.4465), Std (0.2470, 0.2435, 0.2616)


In [ ]:
from src.data import get_cifar10_dataloaders

# Determine batch size according to compute availability
batch_size = 128 if torch.cuda.is_available() else config.CLASSIFIER_BATCH_SIZE

train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    data_dir=config.DATA_DIR,
    batch_size=batch_size,
    val_split_ratio=config.VAL_SPLIT_RATIO,
    random_seed=config.RANDOM_SEED,
    augmentation="standard",
    num_workers=2 if torch.cuda.is_available() else 0,
)

print(f"[Dataset Partitions]")
print(f"  ├── Training Batches:   {len(train_loader)} ({len(train_loader.dataset):,} images)")
print(f"  ├── Validation Batches: {len(val_loader)} ({len(val_loader.dataset):,} images)")
print(f"  └── Test Batches:       {len(test_loader)} ({len(test_loader.dataset):,} images)")


## 6. Module 1: Baseline Custom CNN Classifier Training (Resumable)
- **Architecture:** Preserved 3-Block Custom CNN (`CIFAR10Classifier`).
- **Schedule:** 15 Epochs, Adam optimizer ($\\text{lr}=10^{-3}$, $\\text{weight\\_decay}=10^{-4}$), CrossEntropyLoss.
- **Resumable Checkpoints:**
  - `checkpoints/latest_classifier.pth`: Saved after every completed epoch.
  - `checkpoints/baseline_classifier_best.pth`: Saved whenever validation accuracy strictly improves.
- If disconnected, rerun this cell to resume seamlessly from the next unfinished epoch.


In [ ]:
from src.models import CIFAR10Classifier
from src.train import train_classifier, plot_training_history

baseline_classifier = CIFAR10Classifier(num_classes=config.NUM_CLASSES).to(device)

baseline_best_path = config.CHECKPOINT_DIR / config.BASELINE_CLASSIFIER_BEST
baseline_latest_path = config.CHECKPOINT_DIR / config.BASELINE_CLASSIFIER_LATEST
baseline_history_path = config.OUTPUTS_DIR / "classifier_history.json"

# Train classifier with automatic resumption
baseline_history = train_classifier(
    model=baseline_classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=config.CLASSIFIER_EPOCHS,
    lr=config.CLASSIFIER_LR,
    weight_decay=config.CLASSIFIER_WEIGHT_DECAY,
    checkpoint_path=baseline_best_path,
    latest_checkpoint_path=baseline_latest_path,
    resume=True,
    history_json_path=baseline_history_path,
    experiment_id="baseline_classifier",
)

# Plot training curves
plot_training_history(
    history=baseline_history,
    title="Baseline CIFAR-10 Classifier",
    save_path=config.OUTPUTS_DIR / "training_curves.png",
)


## 7. Baseline Classifier Test Set Evaluation
Evaluate the baseline model on the untouched test set (10,000 samples).


In [ ]:
from src.train import evaluate_classifier_test_set

baseline_eval = evaluate_classifier_test_set(
    model=baseline_classifier,
    test_loader=test_loader,
    device=device,
    save_cm_path=config.OUTPUTS_DIR / "confusion_matrix.png",
)

print(f"\n[Baseline Classifier Test Results]")
print(f"  Overall Test Accuracy: {baseline_eval['test_accuracy'] * 100:.2f}%")
print("\nClassification Report:")
for cls_name, metrics in baseline_eval['classification_report'].items():
    if isinstance(metrics, dict):
        print(f"  {cls_name:<12} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f} | F1: {metrics['f1-score']:.4f}")


## 8. Controlled CNN Classifier Optimization Experiments (Phase 6)
### Preserving the Custom CNN Architecture
**Requirement:** Retain the custom 3-block CNN architecture without replacing it with ResNet, VGG, ViT, or external backbones.

### Investigation of Baseline Accuracy Gap
In the baseline model, training accuracy (70.95%) was noticeably lower than validation accuracy (78.38%).
- **Root Cause:** In `CIFAR10Classifier`, `Dropout2d(0.25)` is active across all conv blocks, and `Dropout(0.50)` is active in the dense head during training (`model.train()`). During validation (`model.eval()`), dropout is deactivated and all weights are active. Furthermore, random cropping and horizontal flips augment the training data.

### Controlled Optimization Experiments
| Experiment | Modifications | Target |
| :--- | :--- | :--- |
| **Experiment A** | Enhanced data augmentation (RandomCrop + Flip + mild ColorJitter + Rotation) | Test generalization |
| **Experiment B** | Learning Rate Scheduler (`CosineAnnealingLR`) | Fine-grained convergence |
| **Experiment C** | Regularization tuning (Label Smoothing 0.05 + tuned weight decay) | Calibration & overconfidence mitigation |
| **Experiment D** | Combined Best Synergistic Configuration | Target: **85%+ validation accuracy** |

We execute the combined best experiment below with dedicated checkpoints (`optimized_classifier_best.pth` and `optimized_classifier_latest.pth`).


In [ ]:
from src.data import get_cifar10_dataloaders
from src.train import train_classifier, plot_training_history, evaluate_classifier_test_set

opt_config = config.OPTIMIZATION_EXPERIMENTS["experiment_d"]

# 1. Prepare data loaders with enhanced augmentation
opt_train_loader, opt_val_loader, opt_test_loader = get_cifar10_dataloaders(
    data_dir=config.DATA_DIR,
    batch_size=batch_size,
    val_split_ratio=config.VAL_SPLIT_RATIO,
    random_seed=config.RANDOM_SEED,
    augmentation=opt_config["augmentation"],
    num_workers=2 if torch.cuda.is_available() else 0,
)

# 2. Instantiate preserved Custom CNN architecture
optimized_classifier = CIFAR10Classifier(num_classes=config.NUM_CLASSES).to(device)

opt_best_path = config.CHECKPOINT_DIR / config.OPTIMIZED_CLASSIFIER_BEST
opt_latest_path = config.CHECKPOINT_DIR / config.OPTIMIZED_CLASSIFIER_LATEST
opt_history_path = config.OUTPUTS_DIR / "optimization_results.json"

# 3. Train optimized classifier
optimized_history = train_classifier(
    model=optimized_classifier,
    train_loader=opt_train_loader,
    val_loader=opt_val_loader,
    device=device,
    epochs=opt_config["epochs"],
    lr=opt_config["lr"],
    weight_decay=opt_config["weight_decay"],
    checkpoint_path=opt_best_path,
    latest_checkpoint_path=opt_latest_path,
    scheduler_type=opt_config["scheduler"],
    scheduler_params=opt_config.get("scheduler_params"),
    label_smoothing=opt_config.get("label_smoothing", 0.0),
    resume=True,
    history_json_path=opt_history_path,
    experiment_id="optimized_classifier_exp_d",
)

# 4. Evaluate optimized classifier on untouched test set
optimized_eval = evaluate_classifier_test_set(
    model=optimized_classifier,
    test_loader=opt_test_loader,
    device=device,
    save_cm_path=config.OUTPUTS_DIR / "optimized_confusion_matrix.png",
)

plot_training_history(
    history=optimized_history,
    title="Optimized CIFAR-10 Classifier (Exp D)",
    save_path=config.OUTPUTS_DIR / "optimized_training_curves.png",
)

print(f"\n[Optimized Classifier Results]")
print(f"  Best Val Accuracy:  {max(optimized_history['val_acc']) * 100:.2f}%")
print(f"  Test Accuracy:      {optimized_eval['test_accuracy'] * 100:.2f}%")


## 9. Optimization Comparison Summary
Compares historical baseline, reproduced baseline, and optimized CNN results.


In [ ]:
import json

baseline_best_val = max(baseline_history['val_acc']) if baseline_history['val_acc'] else 0.0
opt_best_val = max(optimized_history['val_acc']) if optimized_history['val_acc'] else 0.0

comparison_data = {
    "historical_baseline": {
        "architecture": config.HISTORICAL_CLASSIFIER["architecture"],
        "epochs": config.HISTORICAL_CLASSIFIER["epochs"],
        "val_accuracy": config.HISTORICAL_CLASSIFIER["final_val_acc"],
        "test_accuracy": "Not available (lost checkpoint)",
    },
    "reproduced_baseline": {
        "architecture": "Custom CIFAR10Classifier",
        "epochs": config.CLASSIFIER_EPOCHS,
        "val_accuracy": baseline_best_val,
        "test_accuracy": baseline_eval['test_accuracy'],
    },
    "optimized_cnn": {
        "architecture": "Custom CIFAR10Classifier (Preserved)",
        "epochs": opt_config['epochs'],
        "val_accuracy": opt_best_val,
        "test_accuracy": optimized_eval['test_accuracy'],
    }
}

print("=" * 75)
print(f"{'Experiment':<22} | {'Architecture':<24} | {'Val Acc':<10} | {'Test Acc':<10}")
print("-" * 75)
print(f"{'Historical Baseline':<22} | {'Custom CNN':<24} | {config.HISTORICAL_CLASSIFIER['final_val_acc']*100:.2f}%     | {'N/A':<10}")
print(f"{'Reproduced Baseline':<22} | {'Custom CNN':<24} | {baseline_best_val*100:.2f}%     | {baseline_eval['test_accuracy']*100:.2f}%")
print(f"{'Optimized CNN (Exp D)':<22} | {'Custom CNN':<24} | {opt_best_val*100:.2f}%     | {optimized_eval['test_accuracy']*100:.2f}%")
print("=" * 75)

with open(config.OUTPUTS_DIR / "classifier_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison_data, f, indent=2)


## 10. Module 2: FGSM Adversarial Attack Evaluation
### Mathematical Formulation
$$\\mathbf{x}_{adv} = \\text{clip}\\left(\\mathbf{x}_{raw} + \\epsilon \\cdot \\text{sign}\\left(\\nabla_{\\mathbf{x}_{raw}} \\mathcal{L}(f(\\mathbf{x}_{norm}), y)\\right), 0.0, 1.0\\right)$$
Perturbations are performed strictly in the unnormalized $[0, 1]$ physical pixel space to guarantee exact $\\epsilon$ bounds.


In [ ]:
from src.attacks import fgsm_attack, visualize_adversarial_examples, evaluate_fgsm_spectrum

# Select a clean sample from test loader
sample_images, sample_labels = next(iter(test_loader))
sample_idx = 0
sample_img = sample_images[sample_idx]
sample_lbl = sample_labels[sample_idx].item()

fig = visualize_adversarial_examples(
    model=baseline_classifier,
    sample_image=sample_img,
    sample_label=sample_lbl,
    epsilons=[0.01, 0.03, 0.05, 0.10],
    device=device,
    save_path=config.OUTPUTS_DIR / "fgsm_visual_comparison.png",
)


In [ ]:
import json

# Evaluate Attack Success Rate across epsilon spectrum
robustness_results = evaluate_fgsm_spectrum(
    model=baseline_classifier,
    dataloader=test_loader,
    epsilons=config.FGSM_EPSILONS,
    device=device,
    save_plot_path=config.OUTPUTS_DIR / "robustness_results.png",
)

with open(config.OUTPUTS_DIR / "robustness_results.json", "w", encoding="utf-8") as f:
    json.dump(robustness_results, f, indent=2)


## 11. Module 3: Adversarial Detector Dataset Generation & Resumable Training
### Preserving the Adversarial Detector Architecture
- **Detector CNN:** 3 Convolutional Blocks + Dense Binary Classification Head (`AdversarialDetector`).
- **Input:** Images in normalized CIFAR-10 space.
- **Output:** Binary classification (0: Clean Image, 1: Adversarially Perturbed Image).
- **Target Classifier Compatibility:** Generated from the classifier's gradients; train, val, and test splits remain strictly isolated to prevent data leakage.


In [ ]:
from src.train import build_detector_dataset

print("Generating balanced clean vs adversarial dataset for detector training...")

# Detector training split (from classifier train_loader)
detector_train_loader = build_detector_dataset(
    classifier=baseline_classifier,
    clean_loader=train_loader,
    epsilon=config.DEFAULT_EPSILON,
    device=device,
    max_clean_samples=10000,
)

# Detector validation split (from classifier val_loader)
detector_val_loader = build_detector_dataset(
    classifier=baseline_classifier,
    clean_loader=val_loader,
    epsilon=config.DEFAULT_EPSILON,
    device=device,
    max_clean_samples=2500,
)

# Detector test split (from classifier test_loader)
detector_test_loader = build_detector_dataset(
    classifier=baseline_classifier,
    clean_loader=test_loader,
    epsilon=config.DEFAULT_EPSILON,
    device=device,
    max_clean_samples=2500,
)

print(f"[Detector Data Created]")
print(f"  ├── Train pairs: {len(detector_train_loader.dataset):,} samples")
print(f"  ├── Val pairs:   {len(detector_val_loader.dataset):,} samples")
print(f"  └── Test pairs:  {len(detector_test_loader.dataset):,} samples")


In [ ]:
from src.models import AdversarialDetector
from src.train import train_detector, plot_training_history

detector = AdversarialDetector().to(device)

detector_best_path = config.CHECKPOINT_DIR / config.BASELINE_DETECTOR_BEST
detector_latest_path = config.CHECKPOINT_DIR / config.BASELINE_DETECTOR_LATEST
detector_history_path = config.OUTPUTS_DIR / "detector_history.json"

detector_history = train_detector(
    detector=detector,
    train_loader=detector_train_loader,
    val_loader=detector_val_loader,
    device=device,
    epochs=config.DETECTOR_EPOCHS,
    lr=config.DETECTOR_LR,
    weight_decay=config.DETECTOR_WEIGHT_DECAY,
    checkpoint_path=detector_best_path,
    latest_checkpoint_path=detector_latest_path,
    threshold=config.DETECTOR_THRESHOLD,
    resume=True,
    history_json_path=detector_history_path,
    experiment_id="baseline_detector",
)

plot_training_history(
    history=detector_history,
    title="Adversarial Detector Training",
    save_path=config.OUTPUTS_DIR / "detector_training_curves.png",
)


## 12. Quantitative Detector Evaluation (ROC-AUC & Error Metrics)
Calculates Accuracy, Precision, Recall, F1-Score, False Positive Rate (FPR), False Negative Rate (FNR), and ROC-AUC.


In [ ]:
from src.train import evaluate_detector_test_set
import json

det_eval = evaluate_detector_test_set(
    detector=detector,
    test_loader=detector_test_loader,
    device=device,
    threshold=config.DETECTOR_THRESHOLD,
    save_cm_path=config.OUTPUTS_DIR / "detector_confusion_matrix.png",
    save_roc_path=config.OUTPUTS_DIR / "detector_roc_curve.png",
)

print("\n" + "=" * 50)
print("         ADVERSARIAL DETECTOR TEST REPORT        ")
print("=" * 50)
print(f"Accuracy:                 {det_eval['accuracy'] * 100:.2f}%")
print(f"Precision:                {det_eval['precision'] * 100:.2f}%")
print(f"Recall (TPR):             {det_eval['recall'] * 100:.2f}%")
print(f"F1-Score:                 {det_eval['f1_score']:.4f}")
print(f"False Positive Rate (FPR):{det_eval['false_positive_rate'] * 100:.2f}%")
print(f"False Negative Rate (FNR):{det_eval['false_negative_rate'] * 100:.2f}%")
print(f"ROC-AUC Score:            {det_eval['roc_auc']:.4f}")
print("=" * 50)

with open(config.OUTPUTS_DIR / "detector_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(det_eval, f, indent=2)


## 13. Module 4: Integrated ModelShield AI Firewall & Defense Evaluation
### Real-World Protective Utility Benchmark
We benchmark the end-to-end protective utility of ModelShield:
1. **Unprotected Classifier:** Accuracy under adversarial attack.
2. **Protected ModelShield Pipeline:**
   - Adversarial images screened and neutralized before classification.
   - Attack Threat Mitigation Rate (% of attacks blocked).
   - Clean Utility Retention (% of clean images passed through and correctly classified).
   - False Alarm Rate (% of clean images blocked).


In [ ]:
from src.train import evaluate_firewall_defense
import json

defense_results = evaluate_firewall_defense(
    classifier=baseline_classifier,
    detector=detector,
    clean_loader=test_loader,
    epsilon=config.DEFAULT_EPSILON,
    threshold=config.DETECTOR_THRESHOLD,
    device=device,
    max_samples=1000,
)

print("\n" + "=" * 60)
print("       MODELSHIELD END-TO-END DEFENSE BENCHMARK         ")
print("=" * 60)
print(f"Perturbation Magnitude (\\epsilon):           {defense_results['epsilon']:.2f}")
print(f"Firewall Detection Cutoff:              {defense_results['threshold']:.2f}")
print(f"Evaluated Samples:                      {defense_results['total_evaluated_clean']} Clean + {defense_results['total_evaluated_adversarial']} Adversarial")
print("-" * 60)
print(f"Unprotected Classifier Clean Accuracy:   {defense_results['unprotected_clean_accuracy'] * 100:.2f}%")
print(f"Unprotected Accuracy Under Attack:      {defense_results['unprotected_adversarial_accuracy'] * 100:.2f}%")
print("-" * 60)
print(f"Attack Threat Mitigation Rate (Blocked): {defense_results['attack_threat_mitigation_rate'] * 100:.2f}%")
print(f"Clean Utility Retained (Allowed & Corr): {defense_results['clean_utility_retained'] * 100:.2f}%")
print(f"False Alarm Rate (Clean Blocked):       {defense_results['false_alarm_rate'] * 100:.2f}%")
print("=" * 60)

with open(config.OUTPUTS_DIR / "defense_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(defense_results, f, indent=2)


## 14. Live Firewall Inspection Demo
Demonstrates the ModelShield firewall processing individual clean and adversarial inputs.


In [ ]:
from src.firewall import ModelShieldFirewall
from src.attacks import fgsm_attack

firewall = ModelShieldFirewall(
    classifier=baseline_classifier,
    detector=detector,
    threshold=config.DETECTOR_THRESHOLD,
    device=device,
)

# Test 1: Clean Sample
clean_sample = sample_images[0:1].to(device)
clean_result = firewall.process(clean_sample)
print(firewall.format_security_report(clean_result, input_name="clean_sample_cifar10.png"))

# Test 2: Adversarial Sample
adv_sample = fgsm_attack(baseline_classifier, clean_sample, sample_labels[0:1].to(device), epsilon=0.03)
adv_result = firewall.process(adv_sample)
print(firewall.format_security_report(adv_result, input_name="adversarial_attack_eps0.03.png"))


## 15. Permanent Google Drive Storage Verification & Manifest
Verify that all generated weights, structured metrics, and visualization figures exist and possess non-zero file sizes.


In [ ]:
print("=" * 70)
print(f"MODELSHIELD STORAGE MANIFEST: {STORAGE_ROOT}")
print("=" * 70)

for folder_name in ("checkpoints", "outputs", "logs"):
    folder_path = STORAGE_ROOT / folder_name
    print(f"\n[{folder_name.upper()}] -> {folder_path}")
    files = sorted(list(folder_path.glob("*")))
    if not files:
        print("  (No files found yet)")
    for f in files:
        if f.is_file() and not f.name.endswith(".gitkeep"):
            size_mb = f.stat().st_size / (1024 * 1024)
            size_str = f"{size_mb:.2f} MB" if size_mb >= 0.01 else f"{f.stat().st_size} bytes"
            status = "✓ VALID" if f.stat().st_size > 0 else "✗ EMPTY"
            print(f"  ├── {f.name:<35} ({size_str:<10}) [{status}]")

print("\n" + "=" * 70)
print("[✓] Execution and persistence verification complete.")
print("=" * 70)
